<a href="https://colab.research.google.com/github/Anthei0774/Advent-of-Code/blob/main/2023/Day_19_Aplenty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
puzzle = '''px{a<2006:qkq,m>2090:A,rfg}
pv{a>1716:R,A}
lnx{m>1548:A,A}
rfg{s<537:gd,x>2440:R,A}
qs{s>3448:A,lnx}
qkq{x<1416:A,crn}
crn{x>2662:A,R}
in{s<1351:px,qqz}
qqz{s>2770:qs,m<1801:hdj,R}
gd{a>3333:R,R}
hdj{m>838:A,pv}

{x=787,m=2655,a=1222,s=2876}
{x=1679,m=44,a=2067,s=496}
{x=2036,m=264,a=79,s=2244}
{x=2461,m=1339,a=466,s=291}
{x=2127,m=1623,a=2188,s=1013}'''

with open('19.txt') as f: puzzle = f.read()

workflows, parts = puzzle.split('\n\n')

import re

# processing workflows
workflows = workflows.split('\n')
for i, wf in enumerate(workflows):

    # remove {} symbols
    wf = wf[: -1].split('{')

    # format a single workflow into a list of ['xmas', '<>', value, wf or A or R] items
    wf[1] = wf[1].split(',')
    for j, rule in enumerate(wf[1]):
        if ('<' in rule) or ('>' in rule):
            rule = [rule[0], rule[1], int(rule[2:].split(':')[0]), rule.split(':')[-1]]
        else:
            rule = [None, None, None, rule]
        wf[1][j] = rule

    # end of processing
    workflows[i] = wf

# into dict
workflows = {wf[0] : wf[1] for wf in workflows}
workflows

# processing parts
parts = parts.split('\n')
for i, p in enumerate(parts):
    p = p[1 : -1]
    p = p.split(',')
    p = {category.split('=')[0]: int(category.split('=')[1]) for category in p}
    parts[i] = p
parts

###############################################################################
# PART 1

accepted = []
rejected = []

for p in parts:
    # print('Part', p)

    curr = 'in'
    iter = 0
    while curr not in ['A', 'R']:

        wf = workflows[curr]
        # print('Current workflow', curr, wf)

        for rule in wf:
            category, symbol, value, next = rule

            if category is None:
                assert (symbol is None) and (value is None) and (next is not None)
                curr = next
                break
            elif symbol == '<':
                if p[category] < value:
                    curr = next
                    break
            else:
                assert symbol == '>'
                if p[category] > value:
                    curr = next
                    break

    if curr == 'A':
        accepted.append(p)
        # print('Accepted\n')
    else:
        rejected.append(p)
        # print('Rejected\n')

sum(sum(p.values()) for p in accepted)

###############################################################################
# PART 2

# Step -1: cleanup in workflows, because there is "lnx{m>1548:A,A}" in example which always results in moving to accepted
for wf in workflows:
    next_wfs = list(set(rule[-1] for rule in workflows[wf]))
    if len(next_wfs) == 1:
        next = next_wfs[0]
        workflows[wf] = [[None, None, None, next]]
        # print('Cleaning up workflow', wf)
workflows

# Step 0: another cleanup in workflows, because there is "bl{m<1269:A,s>3432:A,s>3277:A,cml}" which has multiple ways to moving to accepted
new_workflows = {}

for wf in workflows:

    # calculate same outcomes
    next_wfs = [rule[-1] for rule in workflows[wf]]
    next_wfs_cnts = {next: next_wfs.count(next) for next in next_wfs}

    # print('Before', wf, workflows[wf])
    for next, cnt in next_wfs_cnts.items():
        if cnt > 1:



            c = 0
            for i, rule in enumerate(workflows[wf]):
                if rule[-1] == next:
                    new = next + str(c)
                    rule[-1] = new
                    c += 1
                    if new not in new_workflows: new_workflows[new] = [[None, None, None, next]]
                workflows[wf][i] = rule

            assert c == cnt

    # print('After', wf, workflows[wf], '\n')

for wf in new_workflows:
    assert wf not in workflows
    workflows[wf] = new_workflows[wf]

workflows

# Step 1: generate a list of paths of workflows which result in a part accepted
to_expand = [['in']]
paths = []

while True:
    # print('Expanding', to_expand)

    next_iter = []
    for p in to_expand:
        wf = p[-1]

        next = workflows[wf]
        next = [n[-1] for n in next]

        for n in next: next_iter.append(p + [n])

    # completed paths
    for p in next_iter:
        if p[-1] in ['A', 'R']:
            paths.append(p)

    # paths to be searched
    to_expand = [p for p in next_iter if p[-1] not in ['A', 'R']]
    if to_expand == []: break


paths = [p for p in paths if p[-1] == 'A']
paths = list(set(tuple(p) for p in paths))
paths = sorted(paths, key=lambda p: len(p))
paths

# Step 2: from each path, calculate the value ranges of a part for which that part is accepted along the path of workflows
paths = {p: {'x': (1, 4000), 'm': (1, 4000), 'a': (1, 4000), 's': (1, 4000)} for p in paths}
paths

for p in paths:
    # print('Path', p)

    for i in range(len(p) - 1):
        # print(p[i], p[i + 1])

        # PLAN
        # to reach p[i + 1] from p[i], we need to find the rule in p[i], fulfill that, while failing every other rule before
        wf = workflows[p[i]]
        # print('Workflow', wf)

        # finding index
        J = [rule[-1] for rule in wf].index(p[i + 1])

        # failing everything before
        for j in range(J):
            # print('Failing', wf[j])

            category, symbol, value, next = wf[j]
            assert next != p[i + 1] and category is not None

            if symbol == '<': # value <= interval
                if paths[p][category][0] < value:
                    paths[p][category] = (value, paths[p][category][1])
            else: # interval <= value
                if value < paths[p][category][1]:
                    paths[p][category] = (paths[p][category][0], value)

        # fulfilling rule J
        # print('Fulfilling', wf[J])
        category, symbol, value, next = wf[J]
        assert next == p[i + 1]

        if category is not None:
            if symbol == '<':
                if value <= paths[p][category][1]:
                    paths[p][category] = (paths[p][category][0], value - 1)
            else:
                if paths[p][category][0] <= value:
                    paths[p][category] = (value + 1, paths[p][category][1])

        # print(paths[p])

    # checking that final path is consistent
    for category in paths[p]:
        assert paths[p][category][0] <= paths[p][category][1]
    # print(paths[p], '\n')

# Step 3: sum up the ranges multiplied
part_ranges = list(paths.values())

S = 0
for p in part_ranges:
    M = 1
    for category in p:
        m = p[category][1] - p[category][0] + 1
        M *= m
    S += M
S